# Notebook 01c — PPO baseline on `CartPole-v1`

Train the standard PPO baseline for CartPole using `config/envs/cartpole.yaml`.
CartPole is a dense-reward implementation self-test: PPO should learn quickly,
and failures usually point to trainer, scaler, success-metric, or checkpointing
bugs rather than sparse exploration.

Important semantics: `terminated` means failure in CartPole. A solved episode is
surviving to the 500-step time limit, which Gymnasium reports as `truncated`.
The config therefore uses `success_on: truncated`.


---
## Knobs


In [ ]:
# ----------------------------------------------------------------------------
# EDIT ME
# ----------------------------------------------------------------------------
ENV_CONFIG    = "cartpole"
FORCE_RETRAIN = True

OVERRIDES = {
    # "ppo.total_timesteps": 50_000,    # quick smoke run
    # "run.seeds": (0,),                # one seed while iterating
    # "run.run_name": "cartpole_smoke",
}
# ----------------------------------------------------------------------------


## 0. Setup


In [ ]:
%load_ext autoreload
%autoreload 2

import sys, pathlib, time

ROOT = pathlib.Path.cwd()
if not (ROOT / "config").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gymnasium as gym

from config import make_config, seed_dir, RUNS_DIR, FIGURES_DIR
from dataio import load_trajectories, validate, load_checkpoint, list_checkpoints
from utils.logging import read_scalars
from utils.plotting import plot_learning_curves, plot_episode_returns, plot_grid, savefig
from scripts.train import run_seeds

cfg = make_config(ENV_CONFIG, **OVERRIDES)
RUN_NAME = cfg.run.run_name
SEEDS = tuple(cfg.run.seeds)
FIG = FIGURES_DIR / f"nb01_{RUN_NAME}"
FIG.mkdir(parents=True, exist_ok=True)

print("project root:", ROOT)
print(cfg.summary())


## 1. Train


In [ ]:
already = all((seed_dir(RUN_NAME, s) / "scalars.csv").exists() for s in SEEDS)

if already and not FORCE_RETRAIN:
    print(f"found existing run at {RUNS_DIR / RUN_NAME} -- skipping training")
    print("set FORCE_RETRAIN = True to re-run")
    results = None
else:
    t0 = time.time()
    results = run_seeds(cfg)
    print()
    print(f"total wall time: {(time.time() - t0) / 60:.1f} min")


## 2. Load Outputs


In [ ]:
scalars  = {s: read_scalars(seed_dir(RUN_NAME, s) / "scalars.csv") for s in SEEDS}
episodes = {s: pd.read_csv(seed_dir(RUN_NAME, s) / "episodes.csv") for s in SEEDS}
trajs    = ({s: load_trajectories(seed_dir(RUN_NAME, s) / "trajectories.npz") for s in SEEDS}
            if cfg.run.record_trajectories else {})

summary = pd.DataFrame({
    "first_time_limit": {s: (int(e.loc[e["success"], "global_step"].iloc[0])
                             if e["success"].any() else None) for s, e in episodes.items()},
    "final_return":     {s: d["mean_return_100"].iloc[-1] for s, d in scalars.items()},
    "final_success":    {s: d["success_rate_100"].iloc[-1] for s, d in scalars.items()},
    "final_entropy":    {s: d["entropy"].iloc[-1] for s, d in scalars.items()},
    "final_expl_var":   {s: d["explained_variance"].iloc[-1] for s, d in scalars.items()},
    "episodes":         {s: len(e) for s, e in episodes.items()},
})
summary.index.name = "seed"
display(summary.round(3))


## 3. Return And Success

Return is episode length, capped at 500. `success_rate_100` is the fraction of
recent episodes that reached the 500-step time limit.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
plot_episode_returns(episodes, window=50, ax=axes[0])
plot_learning_curves(scalars, y="mean_return_100", ylabel="return", title="Return", ax=axes[1])
plot_learning_curves(scalars, y="success_rate_100", ylabel="time-limit rate", title="Success", ax=axes[2])
axes[1].axhline(475, color="#888", ls=":", lw=1)
axes[2].set_ylim(-0.05, 1.05)
fig.tight_layout()
savefig(fig, FIG / "return_and_success.png")
plt.show()


## 4. Optimisation Diagnostics


In [ ]:
keys = [k for k in [
    "entropy", "approx_kl", "clipfrac", "adv_std_raw",
    "explained_variance", "pg_loss", "v_loss", "grad_norm_actor", "grad_norm_critic",
] if any(k in d.columns for d in scalars.values())]
fig = plot_grid(scalars, keys, ncols=3, figsize=(14, 9))
savefig(fig, FIG / "diagnostics.png")
plt.show()


## 5. Checkpoints And Greedy Evaluation


In [ ]:
def greedy_eval(ck, n_episodes=50, seed0=90_000):
    env = gym.make("CartPole-v1")
    rets, succ, lens = [], 0, []
    for ep in range(n_episodes):
        obs, _ = env.reset(seed=seed0 + ep)
        total, t = 0.0, 0
        while True:
            a = int(np.argmax(ck.probs(np.asarray(obs)[None, :])[0]))
            obs, r, term, trunc, _ = env.step(a)
            total += r
            t += 1
            if trunc:
                succ += 1
                break
            if term:
                break
        rets.append(total)
        lens.append(t)
    env.close()
    return float(np.mean(rets)), succ / n_episodes, float(np.mean(lens))

rows = []
for s in SEEDS:
    for p in list_checkpoints(seed_dir(RUN_NAME, s) / "checkpoints"):
        ck = load_checkpoint(p)
        ret, sr, ln = greedy_eval(ck)
        rows.append({"seed": s, "frac": f"{ck.fraction:.0%}", "step": ck.global_step,
                     "greedy_return": round(ret, 1), "greedy_success": sr,
                     "greedy_length": round(ln, 1)})

FRAC_ORDER = [f"{f:.0%}" for f in cfg.run.checkpoint_fractions]
evaldf = pd.DataFrame(rows)
evaldf["frac"] = pd.Categorical(evaldf["frac"], categories=FRAC_ORDER, ordered=True)
display(evaldf.pivot(index="seed", columns="frac", values=["greedy_return", "greedy_success"]))
evaldf


## 6. Trajectory Validation


In [ ]:
all_clean = True
for s, t in trajs.items():
    p = seed_dir(RUN_NAME, s) / "trajectories.npz"
    print(f"seed {s}  ({p.stat().st_size / 1e6:.1f} MB)")
    problems = validate(t, n_actions=t.n_actions)
    print("  ->", problems if problems else "NO PROBLEMS")
    all_clean &= not problems
    print()
print("TRAJECTORY VALIDATION:", "PASS" if all_clean else "FAIL")


## 7. Gate 1 Verdict


In [ ]:
best_greedy = {s: evaldf[evaldf.seed == s]["greedy_return"].max() for s in SEEDS}
checks = {
    "every seed reaches the 500-step time limit during training":
        all(episodes[s]["success"].any() for s in SEEDS),
    "every seed reaches greedy return >= 475 at some checkpoint":
        all(best_greedy[s] >= 475 for s in SEEDS),
    "critic is informative (median explained variance > 0.3)":
        all(scalars[s]["explained_variance"].median() > 0.3 for s in SEEDS),
    "trajectory files reload and validate": all_clean,
}
width = max(len(k) for k in checks)
for k, v in checks.items():
    print(f"  {'PASS' if v else 'FAIL'}  {k:<{width}}")
print()
print("GATE 1:", "PASS -- baseline learns and the dataset is sound"
      if all(checks.values()) else "FAIL -- inspect diagnostics before PPO-CF")
